# clear breakdown of categories

In [3]:
import pandas as pd
import panel as pn

# Load financial rules
def load_data():
    return pd.read_excel("corrected_financial_analytics.xlsx")

# Budget Calculation
def calculate_budget(salary, marital_status, region, num_children, category_rules):
    category_row = category_rules[(category_rules['Region'] == region) & (category_rules['Category'] == marital_status)]
    if category_row.empty:
        needs_percentage, wants_percentage, savings_percentage = 50, 30, 20
    else:
        needs_percentage, wants_percentage, savings_percentage = map(lambda x: sum(map(int, x.strip('%').split('-'))) / 2,
                                                                    category_row.iloc[0][['Needs (%)', 'Wants (%)', 'Savings (%)']])
    if marital_status in ["Married", "Divorced"] and num_children:
        needs_percentage += num_children * 5
        wants_percentage -= num_children * 5
    needs_amount = salary * (needs_percentage / 100)
    wants_amount = salary * (wants_percentage / 100)
    savings_amount = salary * (savings_percentage / 100)
    return needs_amount, wants_amount, savings_amount

# UI Components
category_rules = load_data()
pn.extension()

salary_input = pn.widgets.FloatInput(name="Salary (₹)")
marital_status_input = pn.widgets.RadioButtonGroup(name="Marital Status", options=["Single", "Married", "Divorced"])
region_input = pn.widgets.RadioButtonGroup(name="Region", options=["Metro", "Urban", "Semi-Urban", "Rural"])
num_children_input = pn.widgets.IntInput(name="Number of Children", value=0)
house_type_input = pn.widgets.Select(name="House Type", options=["Own", "Rent"])
house_expense_input = pn.widgets.FloatInput(name="Total House Expenses (₹)", visible=False)
rent_amount_input = pn.widgets.FloatInput(name="Rent Amount (₹)", visible=False)

# Loans Section
loans_question = pn.pane.Markdown("### Do you have Loans?")
has_loans_input = pn.widgets.RadioButtonGroup(name="Do you have Loans?", options=["Yes", "No"])
num_loans_input = pn.widgets.IntInput(name="Number of Loans", value=0, visible=False)
loan_inputs = pn.Column()

transport_expense_input = pn.widgets.FloatInput(name="Transport Expense (₹)")
food_expense_input = pn.widgets.FloatInput(name="Food Expense (₹)")
medical_expense_input = pn.widgets.FloatInput(name="Medical Expense (₹)")

calculate_button = pn.widgets.Button(name="Calculate Budget", button_type='primary')
output_area = pn.pane.Markdown()
output_area_table = pn.Column()

def update_loan_widgets(event):
    loan_inputs.clear()
    if has_loans_input.value == "Yes":
        num_loans_input.visible = True
        for i in range(num_loans_input.value):
            loan_inputs.append(pn.widgets.FloatInput(name=f"Loan {i+1} Amount (₹, Monthly)"))
            loan_inputs.append(pn.widgets.IntInput(name=f"Loan {i+1} Tenure (Years)"))
    else:
        num_loans_input.visible = False

def update_house_expense(event):
    if house_type_input.value == "Own":
        house_expense_input.visible = True
        house_expense_input.name = "Total House Expenses (₹)"
        rent_amount_input.visible = False
    elif house_type_input.value == "Rent":
        rent_amount_input.visible = True
        rent_amount_input.name = "Rent Amount (₹)"
        house_expense_input.visible = False

has_loans_input.param.watch(update_loan_widgets, 'value')
num_loans_input.param.watch(update_loan_widgets, 'value')
house_type_input.param.watch(update_house_expense, 'value')

def on_calculate(event):
    # Calculate budget amounts
    needs, wants, savings = calculate_budget(
        salary_input.value, marital_status_input.value, region_input.value, num_children_input.value, category_rules
    )

    # Define financial breakdowns
    savings_breakdown = pd.DataFrame({
        "Category": ["Short-Term Savings", "Long-Term Savings"],
        "Amount (₹)": [savings * 0.4, savings * 0.6]
    })
    
    short_term_savings = pd.DataFrame({
        "Sub-Category": ["Down Payment", "Emergency Fund", "Education"],
        "Amount (₹)": [savings * 0.15, savings * 0.15, savings * 0.1]
    })
    
    long_term_savings = pd.DataFrame({
        "Sub-Category": ["Retirement Fund", "Life Insurance", "Children Higher Education", "Wealth Creation Investments"],
        "Amount (₹)": [savings * 0.2, savings * 0.1, savings * 0.15, savings * 0.15]
    })
    
    # Define Needs Breakdown with Healthcare Sub-Categories
    needs_breakdown_data = {
        "Sub-Category": ["Housing", "Utilities", "Groceries", "Transportation"],
        "Amount (₹)": [needs * 0.3, needs * 0.2, needs * 0.2, needs * 0.2]
    }

    # Add Healthcare breakdown (sub-categories)
    healthcare_total = needs * 0.1
    needs_breakdown_data["Sub-Category"].append("Healthcare")
    needs_breakdown_data["Amount (₹)"].append(healthcare_total)

    healthcare_subcategories = {
        "Sub-Category": ["Medical Insurance", "Doctor Visits", "Medications", "Emergency Fund"],
        "Amount (₹)": [healthcare_total * 0.4, healthcare_total * 0.3, healthcare_total * 0.2, healthcare_total * 0.1]
    }

    # Add Healthcare Subcategories to the needs breakdown
    needs_breakdown_data["Sub-Category"].extend(healthcare_subcategories["Sub-Category"])
    needs_breakdown_data["Amount (₹)"].extend(healthcare_subcategories["Amount (₹)"])

    # Add Bills section if House Type is "Own"
    if house_type_input.value == "Own":
        needs_breakdown_data["Sub-Category"].append("Bills")
        needs_breakdown_data["Amount (₹)"].append(house_expense_input.value if house_expense_input.value else 0)

    needs_breakdown = pd.DataFrame(needs_breakdown_data)

    wants_breakdown = pd.DataFrame({
        "Sub-Category": ["Dining Out", "Entertainment", "Travel", "Shopping"],
        "Amount (₹)": [wants * 0.25, wants * 0.25, wants * 0.25, wants * 0.25]
    })

    # Summary text
    summary_text = f"""
    ## Budget Plan
    *Marital Status:* {marital_status_input.value}  
    *Region:* {region_input.value}  
    *Salary:* ₹{salary_input.value:,.2f}  

    *Needs Amount:* ₹{needs:,.2f}  
    *Wants Amount:* ₹{wants:,.2f}  
    *Savings Amount:* ₹{savings:,.2f}  
    """

    output_area.object = summary_text  # Display textual summary

    # Display tables properly
    output_area_table.objects = [
        pn.pane.Markdown("### Needs Breakdown"), pn.pane.DataFrame(needs_breakdown, width=500),
        pn.pane.Markdown("### Wants Breakdown"), pn.pane.DataFrame(wants_breakdown, width=500),
        pn.pane.Markdown("### Savings Breakdown"), pn.pane.DataFrame(savings_breakdown, width=500),
        pn.pane.Markdown("### Short-Term Savings Breakdown"), pn.pane.DataFrame(short_term_savings, width=500),
        pn.pane.Markdown("### Long-Term Savings Breakdown"), pn.pane.DataFrame(long_term_savings, width=500),
    ]

calculate_button.on_click(on_calculate)

# Dashboard Layout
dashboard = pn.Column(
    "## Financial Budget Planner",
    salary_input, marital_status_input, region_input, num_children_input,
    house_type_input, house_expense_input, rent_amount_input,
    loans_question, has_loans_input, num_loans_input, loan_inputs,
    transport_expense_input, food_expense_input, medical_expense_input,
    calculate_button, output_area, output_area_table  # Text summary + tables
)

dashboard.servable()

Column
    [0] Markdown(str)
    [1] FloatInput(name='Salary (₹)')
    [2] RadioButtonGroup(name='Marital Status', options=['Single', 'Married', ...], value='Single')
    [3] RadioButtonGroup(name='Region', options=['Metro', 'Urban', ...], value='Metro')
    [4] IntInput(name='Number of Children')
    [5] Select(name='House Type', options=['Own', 'Rent'], value='Own')
    [6] FloatInput(name='Total House E..., visible=False)
    [7] FloatInput(name='Rent Amount (₹)', visible=False)
    [8] Markdown(str)
    [9] RadioButtonGroup(name='Do you have Loans?', options=['Yes', 'No'], value='Yes')
    [10] IntInput(name='Number of Loans', visible=False)
    [11] Column()
    [12] FloatInput(name='Transport Expense (₹)')
    [13] FloatInput(name='Food Expense (₹)')
    [14] FloatInput(name='Medical Expense (₹)')
    [15] Button(button_type='primary', name='Calculate Budget')
    [16] Markdown(None)
    [17] Column()